In [2]:
%pip install -q -U langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.26 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.2.18 which is incompatible.
langchain-anthropic 0.3.17 requires langchain-core<1.0.0,>=0.3.68, but you have langchain-core 1.2.18 which is incompatible.
langchain-community 0.3.27 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.2.18 which is incompatible.
langchain-openai 0.3.28 requires langchain-core<1.0.0,>=0.3.68, but you have langchain-core 1.2.18 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.2.18 which is incompatible.
megaparse 0.0.55 requires langchain-core<0.4,>=0.3, but you have langchain-core 1.2.18 which is incompatible.
megaparse 0.0.55 requires unstructured[all-docs]==0.15.0, but you have unstructured 0.10.25 which i

In [3]:
from math import exp
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0, # ตั้งค่าเป็น 0 เพื่อความแม่นยำสูงสุด
    model_kwargs={
        "response_logprobs": True, 
        "logprobs": 1 # ขอค่า logprob ของ token ที่ดีที่สุด 1 อันดับ
    }
)

In [ ]:
def check_allergy_system(menu_item, allergy_list):
    prompt = f"""ตรวจสอบว่าเมนู "{menu_item}" มีส่วนผสมที่คนแพ้ "{allergy_list}" หรือไม่
    กฎการตอบ:
    - ถ้ามี/เสี่ยง ให้ตอบว่า "True"
    - ถ้าไม่มี/ปลอดภัย ให้ตอบว่า "False"
    คำตอบ:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    
    try:
        logprob_data = response.response_metadata.get("logprobs_result", {})
        chosen_logprob = logprob_data["chosen_candidates"]["log_probability"]
        
        # คำนวณค่าความมั่นใจ Confidence Score
        confidence = exp(chosen_logprob) # สูตร P = e^logprob 
    except (KeyError, IndexError, TypeError):
        confidence = 0.0

    answer = response.content.strip().lower()

    # Human-in-the-loop
    if 0.1 <= confidence <= 0.9:
        status = "ต้องให้มนุษย์ตรวจสอบ (Manual Review)"
    elif answer == "true":
        status = "ตรวจพบสารก่อภูมิแพ้! (System Warn)"
    else:
        status = "ปลอดภัย (Passed)"

    return {
        "answer": answer,
        "confidence": f"{confidence:.2%}",
        "status": status
    }

In [15]:
result = check_allergy_system("คราเปรสสลัด", "นมควาย")
print(f"AI ตอบ: {result['answer']} | ความมั่นใจ: {result['confidence']}")
print(f"สรุปสถานะ: {result['status']}")

AI ตอบ: true | ความมั่นใจ: 0.00%
สรุปสถานะ: 🚨 ตรวจพบสารก่อภูมิแพ้! (System Warn)


## เช็คโมเดลที่ใช้ได้

In [ ]:
%pip install google-generativeai

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv()
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("--- กำลังดึงรายชื่อโมเดลที่ใช้งานได้ ---")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
            
    print("------------------------------------------")
except Exception as e:
    print(f"เกิดข้อผิดพลาดในการดึงข้อมูล: {e}")

--- กำลังดึงรายชื่อโมเดลที่คุณใช้งานได้ ---
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-2.5-flash-lite-preview-09-2025
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-2.5-computer-use-preview

C:\Users\User\AppData\Local\Temp\ipykernel_22944\3521784576.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
